In [1]:
!pip3 install google-adk litellm


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

True

## Agent Node

In [ ]:
from google.adk import Agent
from google.adk.models.lite_llm import LiteLlm
from pydantic import BaseModel

class ComplaintInput(BaseModel):
    customer_name: str
    complaint_text: str

class ComplaintOutput(BaseModel):
    ComplaintCategory: str
    response: str


classify_complaint_tool = Agent(name="classify_complaint_tool",
                                model=LiteLlm(model='gpt-3.5-turbo'),
                                instruction="You are a complaint classification tool. Your task is to classify customer complaints into predefined categories based on the complaint text. The categories are: Billing Issue, Product Defect, Delivery Delay, Customer Service, and Other. You will receive a complaint text as input and you need to determine the appropriate category for the complaint and provide a response to the customer based on the category.",
                                input_schema=ComplaintInput,
                                output_schema=ComplaintOutput
                                )

## Function Node

In [ ]:
from google.adk import Event

def route_complain(node_input: ComplaintInput) -> Event:
    return Event(output = node_input, route = node_input.category)

def enrich_data(node_input: ComplaintInput) -> ComplaintOutput :
   node_input.complaint_text = node_input.complaint_text + " Enriched with additional context about the customer's history and preferences."
   return node_input

LlmAgent(name='classify_complaint_tool', description='', rerun_on_resume=False, wait_for_output=False, retry_config=None, timeout=None, input_schema=<class '__main__.ComplaintInput'>, output_schema=<class '__main__.ComplaintOutput'>, state_schema=None, parent_agent=None, sub_agents=[], before_agent_callback=None, after_agent_callback=None, model=LiteLlm(model='gpt-3.5-turbo', llm_client=<google.adk.models.lite_llm.LiteLLMClient object at 0x106cab950>), instruction='You are a complaint classification tool. Your task is to classify customer complaints into predefined categories based on the complaint text. The categories are: Billing Issue, Product Defect, Delivery Delay, Customer Service, and Other. You will receive a complaint text as input and you need to determine the appropriate category for the complaint and provide a response to the customer based on the category.', global_instruction='', static_instruction=None, tools=[], generate_content_config=None, mode=None, parallel_worker=N

## edges -> The wiring

In [ ]:
from google.adk import Workflow

# Pattern1 : Sequential workflow

Workflow(name ="seq",edges=[('START',nodeA, nodeB, nodeC)])


## pattern 2 : branching workflow Branch A -> Router -> Branch B, Branch C

Workflow(name = "branching", edges= [('START', nodeA, router_fn),
                                     (router_fn,{'Router B': nodeB,
                                                 'Router C': nodeC})])


## Pattern3 : Join node

from google.adk.workflow import JoinNode
join = JoinNode(name = "join_node")

Workflow(name = "join_workflow", edges=[('START', nodeA, nodeB, join),
                                        (join, nodeC)])




## Bulding the graph Step by Step

## What we are going to built:

1. We are running furniture company
2. You get complaints -> if you any bussiness
3. Understadn the compalint and route it to appropiate team
4. You can plan for resolution

Graph Design:

START -> classify_agent -> router_fn

        router_fn -> {QUALITY: handle_quality, DELIVERY: handle_delivery, BILLING : handle_billing}

        each_branch -> report_agent -> END



Classify Agent -> AI to classify the complaint

router_fn  -> handlers with Python function

report Agent -> AI agent



In [34]:
import os, asyncio
from google.adk import Agent,Workflow,Event
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types
from pydantic import BaseModel
import json

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')

In [12]:
## Pydantic Schemas


class ComplaintInput(BaseModel):
    product_name: str
    complaint_text: str

class ComplaintCategory(BaseModel):
    product_name: str
    complaint_text: str
    complaint_category: str

class ComplaintResponse(BaseModel):
    response: str


In [37]:
## AI Agents

classify_agent = Agent(name="classify_agent",
                       model=LiteLlm(model='gpt-4o-mini'),
                       instruction="You are a complaint classification agent. Your task is to classify customer complaints into predefined categories based on the complaint text. The categories are: QUALITY, DELIVERY OR   ",
                       input_schema=ComplaintInput,
                       output_schema=ComplaintCategory
                       )

reporter_agent = Agent(name="reporter_agent",
                       model=LiteLlm(model='gpt-4o-mini'),
                        instruction =f'Write short resolution plan for the compaint based on the complaint details provided.',
                          input_schema=ComplaintCategory,
                          output_schema=ComplaintResponse)

In [38]:
## Function Nodes

def router_fn(node_input: ComplaintCategory) -> Event:
    return Event(output = node_input, route = node_input.complaint_category)

def handle_quality(node_input: ComplaintCategory) -> ComplaintCategory:
    # Implement logic to handle quality complaints
    response = f"Handling quality complaint for product {node_input.product_name}. The complaint details are: {node_input.complaint_text}. The resolution plan is to inspect the product and provide a replacement if necessary."
    return node_input

def handle_delivery(node_input: ComplaintCategory) -> ComplaintCategory:
    # Implement logic to handle delivery complaints
    response = f"Handling delivery complaint for product {node_input.product_name}. The complaint details are: {node_input.complaint_text}. The resolution plan is to track the delivery and expedite it if it's delayed."
    return node_input

def handle_other(node_input: ComplaintCategory) -> ComplaintCategory:
    # Implement logic to handle other complaints
    response = f"Handling other complaint for product {node_input.product_name}. The complaint details are: {node_input.complaint_text}. The resolution plan is to review the complaint and provide a customized solution."
    return node_input




In [39]:
## Edge -- the wiring

root_agent = Workflow(name = "complaint_resolution_workflow", edges=[('START', classify_agent, router_fn),
                                                                (router_fn, {'QUALITY': handle_quality,
                                                                            'DELIVERY': handle_delivery,
                                                                            'OTHER': handle_other}),
                                                                (handle_quality, reporter_agent),
                                                                (handle_delivery, reporter_agent),
                                                                (handle_other, reporter_agent)
                                                                ])

In [45]:
## Run the service

session_service = InMemorySessionService()
runner = Runner(agent=root_agent, app_name="complaint_resolution_app", session_service=session_service)

async def run(product_name, complaint_text):
    sess = await session_service.create_session(app_name="complaint_resolution_app",user_id="user_123")
    msg = types.Content(role='user',parts=[types.Part(text = json.dumps({"product_name": product_name, "complaint_text": complaint_text}))])

    async for event in runner.run_async(user_id="user_123", new_message=msg, session_id=sess.id):
        print(f"Event: {event}")
        print("********************************************")

        # if event.is_final_response:
        #     print("Agent response: ", event.message.parts[0].text)


In [ ]:
await run(product_name="Smartphone", complaint_text="The battery life of the smartphone is very poor and it drains quickly.")

Event: model_version='gpt-4o-mini-2024-07-18' content=Content(
  parts=[
    Part(
      text='{"product_name":"Smartphone","complaint_text":"The battery life of the smartphone is very poor and it drains quickly.","complaint_category":"QUALITY"}'
    ),
  ],
  role='model'
) grounding_metadata=None partial=False turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  cached_content_token_count=0,
  candidates_token_count=31,
  prompt_token_count=134,
  total_token_count=165
) live_session_resumption_update=None live_session_id=None go_away=None input_transcription=None output_transcription=None avg_logprobs=None logprobs_result=None cache_metadata=None citation_metadata=None interaction_id=None invocation_id='e-dd07aa7f-807a-42f2-b7c1-02c418798b49' author='classify_agent' actions=EventActions(skip_summarization=None, state_delta={}, artifact_delta={}, tra